# AMR Project - Step 1: Data Cleaning (NCBI Dataset)

This notebook covers the initial data cleaning phase for the NCBI Pathogen Detection antimicrobial resistance dataset (`data/raw/data.csv`).

### Cleaning Steps:
1. **Remove Duplicate Rows**: Check and remove exact duplicate records.
2. **Target Filtering**: Standardize the target variable `Resistance phenotype` to binary classification (`susceptible` vs `resistant`), dropping rows with missing/undefined labels.
3. **Drop High Missingness & Identifiers**: Drop `Disk diffusion (mm)` due to >98% missingness, and remove identifiers like `BioSample`, `Isolate`, and `Create date`.

In [7]:
import os
import pandas as pd
import numpy as np

## 1. Load the Raw Data

In [8]:
raw_data_path = "../data/raw/data.csv"
print(f"Loading raw data from {raw_data_path}...")
df = pd.read_csv(raw_data_path, engine="python", on_bad_lines="skip")
print(f"Raw dataset shape: {df.shape}")

Loading raw data from ../data/raw/data.csv...
Raw dataset shape: (494079, 17)


## 2. Inspect Target Class & Duplicates

In [9]:
print("Resistance phenotype value counts:")
print(df['Resistance phenotype'].value_counts(dropna=False))

print("\nNumber of duplicate rows:", df.duplicated().sum())

Resistance phenotype value counts:
Resistance phenotype
susceptible                   305655
resistant                      96176
not defined                    81394
intermediate                   10252
nonsusceptible                   317
susceptible-dose dependent       285
Name: count, dtype: int64

Number of duplicate rows: 0


## 3. Clean the Data

In [10]:
# Drop duplicates
df_cleaned = df.drop_duplicates()

# Filter target to binary ('susceptible' and 'resistant')
valid_targets = ['susceptible', 'resistant']
df_cleaned = df_cleaned[df_cleaned['Resistance phenotype'].isin(valid_targets)].copy()
print(f"Shape after filtering target: {df_cleaned.shape}")

# Drop identifiers and incomplete columns
drop_cols = ['BioSample', 'Isolate', 'Create date', 'Disk diffusion (mm)']
df_cleaned = df_cleaned.drop(columns=drop_cols)
print(f"Shape after dropping columns: {df_cleaned.shape}")

Shape after filtering target: (401831, 17)
Shape after dropping columns: (401831, 13)


## 4. Verify Remaining Columns & Missing Values

In [11]:
print("Cleaned columns:", df_cleaned.columns.tolist())
print("\nMissing values per column:")
print(df_cleaned.isnull().sum())

Cleaned columns: ['Organism group', 'Scientific name', 'Isolation type', 'Location', 'Isolation source', 'Antibiotic', 'Resistance phenotype', 'Measurement sign', 'MIC (mg/L)', 'Laboratory typing platform', 'Vendor', 'Laboratory typing method version or reagent', 'Testing standard']

Missing values per column:
Organism group                                      0
Scientific name                                     0
Isolation type                                   1086
Location                                        18595
Isolation source                                27240
Antibiotic                                          0
Resistance phenotype                                0
Measurement sign                                 5306
MIC (mg/L)                                      11110
Laboratory typing platform                     137701
Vendor                                         163815
Laboratory typing method version or reagent    214662
Testing standard                        

## 5. Save Cleaned Data

In [12]:
processed_dir = "../data/processed"
os.makedirs(processed_dir, exist_ok=True)
processed_path = os.path.join(processed_dir, "cleaned_data.csv")

print(f"Saving cleaned dataset to {processed_path}...")
df_cleaned.to_csv(processed_path, index=False)
print("Cleaned data saved successfully!")

Saving cleaned dataset to ../data/processed\cleaned_data.csv...
Cleaned data saved successfully!
